# Deploy Text Generation Model sebagai API Endpoint (Part 2)

Lanjutan dari `01_code_generation_from_scratch.ipynb`. Model yang sudah dilatih (disimpan ke Google Drive) dibungkus jadi **REST API** dengan FastAPI, lalu di-expose ke internet lewat **ngrok** supaya bisa diakses dari luar Colab.

Ini adalah jawaban langsung dari instruksi final project: *"Buat 1 Proyek NLP sampai jadi Endpoint"*.

In [1]:
!pip install fastapi uvicorn pyngrok pydantic transformers torch

In [2]:
# Install library tambahan jika belum ada
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from pyngrok import ngrok
import torch
import os
import uvicorn
import threading

## 1. Muat Model Hasil Training

In [3]:
# Mount Google Drive (tempat model hasil training tersimpan)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Path ke model di Google Drive
# Path ini HARUS sama persis dengan save_dir di notebook 01 (cell "Simpan Model ke Google Drive")
model_dir = "/content/drive/MyDrive/text-generation-project/code-generator-model"

import os
if not os.path.exists(model_dir):
    raise FileNotFoundError(
        f"Model tidak ditemukan di '{model_dir}'.\n"
        "Kemungkinan penyebab:\n"
        "  1. Belum menjalankan cell 'Simpan Model ke Google Drive' di notebook 01_code_generation_from_scratch.ipynb\n"
        "  2. Runtime Colab sebelumnya sudah berakhir sebelum sempat menyimpan (Colab me-reset storage lokal tiap sesi baru)\n"
        "  3. Path Drive salah ketik / typo\n"
        "Jalankan ulang training + cell save di notebook 01 terlebih dulu, baru lanjutkan notebook ini."
    )
print(f"Model directory ditemukan: {model_dir}")

# Initialize the device
device = torch.device("cpu")

# Load the model and tokenizer from the Google Drive directory
model = AutoModelForCausalLM.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# Create the pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
    top_k=50,       # Memfilter kandidat output
    top_p=0.9,       # Sampling dengan probabilitas kumulatif
    temperature=0.8  # Mengontrol keacakan
)

Model directory ditemukan: /content/drive/MyDrive/text-generation-project/code-generator-model


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_k', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## 2. Definisikan FastAPI App

In [5]:
# FastAPI app
app = FastAPI(title="Text Generation Model Serving")

# Request body schema
class TextGenerationRequest(BaseModel):
    input_text: str
    num_return_sequences: int = 1

# Response body schema
class TextGenerationResponse(BaseModel):
    generated_text: str

@app.post("/generate", response_model=TextGenerationResponse)
def generate_text(request: TextGenerationRequest):
    """
    Generate text based on the input prompt.
    """
    try:
        # Generate text
        results = pipe(request.input_text, num_return_sequences=request.num_return_sequences)
        # Return the first generated sequence
        return TextGenerationResponse(generated_text=results[0]["generated_text"])
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Text generation failed: {str(e)}")

@app.get("/")
def read_root():
    return {"message": "Welcome to the Text Generation API!"}

## 3. Konfigurasi ngrok

**Catatan keamanan:** jangan pernah hardcode token ngrok langsung di cell (token lama di notebook asli sudah ter-expose plaintext — sudah waktunya di-rotate). Simpan token di **Colab Secrets** (ikon kunci 🔑 di sidebar kiri, key name `NGROK_AUTHTOKEN`) lalu ambil lewat `os.getenv` / `userdata.get`.

In [11]:
from google.colab import userdata

# Ambil token dari Colab Secrets, JANGAN ditulis langsung di sini
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
!ngrok config add-authtoken $NGROK_AUTHTOKEN

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


## 4. Jalankan Server + Tunnel

In [12]:
# Function to run FastAPI server
def run_app():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start ngrok and FastAPI server
ngrok_tunnel = ngrok.connect(8000)
print(f"Public URL: {ngrok_tunnel.public_url}")

# Run FastAPI in a thread
thread = threading.Thread(target=run_app)
thread.start()

Public URL: https://tactile-diploma-dejected.ngrok-free.dev


## 5. Uji Coba Endpoint

In [13]:
import requests

url = f"{ngrok_tunnel.public_url}/generate"

data = {
    "input_text": """\
                  # create some data
                  x = np.random.randn(100)
                  y = np.random.randn(100)

                  # create dataframe from x and y
                  """,
    "num_return_sequences": 1,
}

response = requests.post(url, json=data)
print(response.json())

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


INFO:     34.182.235.137:0 - "POST /generate HTTP/1.1" 200 OK
{'generated_text': '                  # create some data\n                  x = np.random.randn(100)\n                  y = np.random.randn(100)\n\n                  # create dataframe from x and y\n                  \n    # a not a dataset of the a function of the dataset\n    # or a copy of the\ndata = True\n        # for data\n    import scipy.\n    #\n    from the the data\n    """\n    # * return p, n_csv_import sys\nimport matplotlib.path,\n    def __init__(self, y, "0, \'\nfrom ..utils.path import check_toolkits.sparse_state, if return plt.path, 4,\n    """\n    # : data\n    for the dataset_path,\n    def __init__(self,\n    """\n    """\n    _score=None,\n    return np.add_2, if load_model,\n    from sklearn.\n\n    def test_array_data_csv(self, \'class_path.random_samples_model_score, "score):\n        X_path.feature_equal, _data_matrix(self):\n    def test_0, **kwargs\n    assert_utils.data = np.add_range)\n    im

---
*Materi: rubythalib.ai AI Engineer Bootcamp — Modul NLP (mentor: Muhammad Ikhwan Fathulloh). Notebook ini adalah versi konsolidasi/rapi dari catatan latihan sesi Text Generation, disusun ulang untuk dokumentasi pribadi.*